In [2]:
import pandas as pd
from pathlib import Path

proc = Path("..") / "data" / "processed"
e = pd.read_parquet(proc / "empreitadas.parquet")
atos = pd.read_parquet(proc / "modificacoes_atos.parquet")

janela = e[e["ano_celebracao"].between(2019, 2022) & e["modificado"]]
print(f"Empreitadas modificadas 2019-2022: {len(janela)}")

Empreitadas modificadas 2019-2022: 7241


In [3]:
# 1. Como o prazo final se compara ao original nos contratos modificados
razao = janela["prazo_final_mod"] / janela["prazoExecucao"]
print("Prazo final ÷ prazo original:")
print(f"  menor que o original: {(razao < 1).mean():.0%}")
print(f"  igual ao original:    {(razao == 1).mean():.0%}")
print(f"  maior (prorrogado):   {(razao > 1).mean():.0%}")

# 2. A última modificação "esquece" prorrogações anteriores?
#    Comparamos o prazo do último ato com o MAIOR prazo já registado no contrato
maior = atos.groupby("idcontrato")["modifPrazoExecucao"].max().rename("prazo_maximo")
j = janela.merge(maior, on="idcontrato")
esquece = j["prazo_maximo"] > j["prazo_final_mod"]
print(f"\nContratos em que o último ato tem prazo MENOR que um ato anterior: {esquece.mean():.0%} ({esquece.sum()})")

# 3. Os contratos com prorrogação: quanto foi prorrogado
prorrog = janela[razao > 1]
print(f"\nProrrogação nos contratos prorrogados (razão prazo final ÷ original):")
print((prorrog["prazo_final_mod"] / prorrog["prazoExecucao"]).describe(percentiles=[.25, .5, .75, .9]).round(2))

Prazo final ÷ prazo original:
  menor que o original: 2%
  igual ao original:    53%
  maior (prorrogado):   45%

Contratos em que o último ato tem prazo MENOR que um ato anterior: 0% (0)

Prorrogação nos contratos prorrogados (razão prazo final ÷ original):
count    3279.00
mean        1.69
std         1.76
min         1.00
25%         1.19
50%         1.39
75%         1.78
90%         2.44
max        88.50
dtype: float64


In [4]:
oficial = pd.DataFrame({
    "contratos_oficial": [12894, 15215, 10605, 13002],
    "valor_oficial_Meur": [3547.02, 4434.72, 2813.0, 4140.0],
}, index=[2020, 2021, 2022, 2023])

nosso = (e.groupby("ano_celebracao")
          .agg(contratos_nosso=("idcontrato", "size"),
               valor_nosso_Meur=("precoContratual", lambda s: s.sum() / 1e6)))

comp = oficial.join(nosso)
comp["dif_contratos_%"] = (comp["contratos_nosso"] / comp["contratos_oficial"] - 1) * 100
comp["dif_valor_%"] = (comp["valor_nosso_Meur"] / comp["valor_oficial_Meur"] - 1) * 100
comp.round(1)

,contratos_oficial,valor_oficial_Meur,contratos_nosso,valor_nosso_Meur,dif_contratos_%,dif_valor_%
2020,12894,3547.0,12733,3932.6,-1.2,10.9
2021,15215,4434.7,13844,3741.5,-9.0,-15.6
2022,10605,2813.0,9882,2732.5,-6.8,-2.9
2023,13002,4140.0,12461,4561.8,-4.2,10.2


In [5]:
# Peso dos 10 maiores contratos no total de cada ano
for ano in [2020, 2021, 2022, 2023]:
    d = e[e["ano_celebracao"] == ano].nlargest(10, "precoContratual")
    total = e.loc[e["ano_celebracao"] == ano, "precoContratual"].sum()
    print(f"\n===== {ano}: 10 maiores = {d['precoContratual'].sum()/1e6:,.0f} M€ "
          f"({d['precoContratual'].sum()/total:.0%} do total) =====")
    print(d[["idcontrato", "precoContratual", "precoBaseProcedimento", "prazoExecucao", "adjudicante"]]
          .assign(precoContratual=lambda x: (x["precoContratual"]/1e6).round(1),
                  precoBaseProcedimento=lambda x: (x["precoBaseProcedimento"]/1e6).round(1))
          .to_string(index=False))


===== 2020: 10 maiores = 959 M€ (24% do total) =====
 idcontrato  precoContratual  precoBaseProcedimento  prazoExecucao                                                                             adjudicante
    7626845            189.1                  235.0           1278                                                       503278602 - Metro do Porto, S. A.
    8129113            148.9                  150.4            910                               508188423 - Administração Central do Sistema de Saúde, IP
    7348435            130.3                  141.0            900 501449752 - APDL - Administração dos Portos do Douro, Leixões e Viana do Castelo, S. A.
    7626823             99.0                  130.0           1034                                                       503278602 - Metro do Porto, S. A.
    6788702             74.8                   80.0            810                                          503933813 - Infraestruturas de Portugal, S. A.
    7286899     